In [ ]:
# %% [markdown]
# # Chile Mineral Supply Chain Pipeline
#
# Builds a directed supply chain graph for Chile's mineral sector by integrating
# USGS/Sernageomin facility inventories with COCHILCO production and export data.
#
# **Structure:**
#   - Cell 1: Setup (imports, paths, utilities, data loading)
#   - Cell 2: COCHILCO Integration (Cu, Mo production matching + Li fixes)
#   - Cell 3: Downstream Supply Chain (stage classification, smelters, ports, edges)
#   - Cell 4: Export Destinations (COCHILCO Section D, port-to-country edges)
#   - Cell 5: Cleanup & Validation (smelter names, dedup, path tracing)
#   - Cell 6: Diagnostics (single merged audit)
#   - Cell 7: Port Distance Comparison (actual vs optimal)
#
# **Inputs:**
#   - `Preliminary/Chile_Minerals_Inventory.csv` (from Chile_Pipeline.py)
#   - `Preliminary/Chile_Mine_Plant_Links.csv` (from Chile_Pipeline.py)
#   - `data/COCHILCO_Production_2005_2024.xlsx` (extracted yearbook)
#   - Original COCHILCO yearbook XLSX (for Mo parsing)
#
# **Final Outputs:**
#   - `Chile_Minerals_Inventory.csv` (with CHAIN_STAGE, production columns)
#   - `Chile_Mine_Plant_Links.csv` (with PRODUCT_FORM)
#   - `Chile_Downstream_Links.csv`
#   - `Chile_Supply_Chain_Edges.csv` (unified, all 4 layers)
#   - `Chile_Export_Destinations.csv`
#   - `Chile_Ports.csv`

# %% 1. SETUP
import os, shutil, re
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

# ── Paths ──────────────────────────────────────────────────────────────────

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
DIR_PRELIM = os.path.join(BASE_DIR, "Preliminary")
COCHILCO_PATH = os.path.join(BASE_DIR, "data", "COCHILCO_Production_2005_2024.xlsx")

_cochilco_orig_candidates = [
    os.path.join(BASE_DIR, "data", "1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    os.path.join(BASE_DIR, "data", "Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    "/Users/leoss/Downloads/1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
    "/Users/leoss/Downloads/Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
]
COCHILCO_ORIG = next((p for p in _cochilco_orig_candidates if os.path.exists(p)), _cochilco_orig_candidates[0])

# ── Shared constants ───────────────────────────────────────────────────────

COMPANY_TO_DEPOSIT = {
    "División El Teniente": ["El Teniente", "Teniente"],
    "División Chuquicamata": ["Chuquicamata"],
    "División Radomiro Tomic": ["Radomiro Tomic", "Radomiro"],
    "División Andina": ["Andina"],
    "División Ministro Hales": ["Ministro Hales"],
    "División Gabriela Mistral": ["Gabriela Mistral", "Gaby"],
    "División Salvador": ["Salvador"],
    "Escondida": ["Escondida"], "Collahuasi": ["Collahuasi"],
    "Los Pelambres": ["Los Pelambres", "Pelambres"],
    "Spence": ["Spence"], "Quebrada Blanca": ["Quebrada Blanca"],
    "Los Bronces": ["Los Bronces", "Bronces"],
    "Sierra Gorda": ["Sierra Gorda"], "Candelaria": ["Candelaria"],
    "Caserones": ["Caserones"], "Centinela (Súlfuros)": ["Centinela"],
    "El Abra": ["El Abra", "Abra"], "Zaldívar": ["Zaldívar", "Zaldivar"],
    "Antucoya": ["Antucoya"], "Lomas Bayas": ["Lomas Bayas"],
    "Mantoverde": ["Mantoverde"], "El Soldado": ["El Soldado", "Soldado"],
    "Mantos Blancos": ["Mantos Blancos"], "Andacollo": ["Andacollo"],
    "Michilla": ["Michilla"], "Franke": ["Franke"],
    "Mantos de la Luna": ["Mantos de la Luna"],
    "Cerro Negro": ["Cerro Negro"],
    # Cerro Colorado excluded: idle mine, 0 kMT production in 2024
    "Las Cenizas": ["Las Cenizas", "Cenizas", "Las Luces", "Aguilucho"],
    "Tres Valles": ["Tres Valles"],
}

CODELCO_MAP = {
    "División El Teniente":      {"search": ["El Teniente", "Teniente"]},
    "División Chuquicamata":     {"search": ["Chuquicamata"]},
    "División Radomiro Tomic":   {"search": ["Radomiro Tomic", "Radomiro"]},
    "División Andina":           {"search": ["Andina", "Rio Blanco"]},
    "División Ministro Hales":   {"search": ["Ministro Hales", "Mansa Mina"]},
    "División Gabriela Mistral": {"search": ["Gabriela Mistral", "Gaby"]},
    "División Salvador":         {"search": ["Salvador", "El Salvador", "Potrerillos"]},
}

SMELTERS = [
    {"name": "Chuquicamata smelter",
     "search": ["Chuquicamata SX-EW plant (oxide) and smelter", "Chuquicamata Division", "Chuquicamata plant"],
     "operator": "Codelco", "smelter_type": "integrated", "region": "Antofagasta",
     "lat": -22.32, "lon": -68.93, "has_refinery": True,
     "feeds_from_mines": ["Chuquicamata", "Radomiro Tomic", "Ministro Hales"],
     "output_product": "cathode", "export_ports": ["Angamos", "Mejillones"]},
    {"name": "Potrerillos smelter",
     "search": ["Potrerillos SX-EW refinery and smelter", "Potrerillos plant"],
     "operator": "Codelco", "smelter_type": "integrated", "region": "Atacama",
     "lat": -26.39, "lon": -69.46, "has_refinery": True,
     "feeds_from_mines": ["Salvador"],
     "output_product": "cathode", "export_ports": ["Barquito"]},
    {"name": "Caletones smelter",
     "search": ["El Teniente plant"],
     "operator": "Codelco", "smelter_type": "integrated", "region": "O'Higgins",
     "lat": -34.12, "lon": -70.48, "has_refinery": True,
     "feeds_from_mines": ["El Teniente"],
     "output_product": "cathode", "export_ports": ["San Antonio", "Ventanas"]},
    {"name": "Altonorte smelter",
     "search": ["Altonorte"],
     "operator": "Glencore", "smelter_type": "custom", "region": "Antofagasta",
     "lat": -23.78, "lon": -70.31, "has_refinery": False,
     "feeds_from_mines": [], "feeds_from_region": "Antofagasta",
     "output_product": "blister", "export_ports": ["Antofagasta", "Mejillones"]},
    {"name": "Paipote smelter (H.V. Lira)",
     "search": ["Paipote", "Hernan Videla", "Hernán Videla"],
     "operator": "ENAMI", "smelter_type": "custom", "region": "Atacama",
     "lat": -27.37, "lon": -70.30, "has_refinery": False,
     "feeds_from_mines": [], "feeds_from_region": "Atacama",
     "output_product": "blister", "export_ports": ["Barquito", "Caldera"]},
    {"name": "Chagres smelter",
     "search": ["Chagres"],
     "operator": "Anglo American", "smelter_type": "integrated", "region": "Valparaiso",
     "lat": -32.78, "lon": -70.97, "has_refinery": False,
     "feeds_from_mines": ["Los Bronces", "El Soldado"],
     "output_product": "blister", "export_ports": ["Ventanas"]},
]

PORTS = [
    {"name": "Coloso",            "region": "Antofagasta", "lat": -23.76, "lon": -70.45,
     "products": "Cu concentrate", "key_users": "Escondida (dedicated)"},
    {"name": "Angamos",           "region": "Antofagasta", "lat": -23.10, "lon": -70.42,
     "products": "Cu cathode, concentrate", "key_users": "Codelco"},
    {"name": "Mejillones",        "region": "Antofagasta", "lat": -23.10, "lon": -70.45,
     "products": "Cu concentrate, cathode, acid", "key_users": "Multiple"},
    {"name": "Antofagasta (ATI)", "region": "Antofagasta", "lat": -23.65, "lon": -70.40,
     "products": "Cu cathode, general", "key_users": "Antofagasta Minerals"},
    {"name": "Iquique",           "region": "Tarapaca",    "lat": -20.21, "lon": -70.15,
     "products": "Cu cathode", "key_users": "Collahuasi, Quebrada Blanca"},
    {"name": "Patache",           "region": "Tarapaca",    "lat": -20.80, "lon": -70.22,
     "products": "Cu concentrate", "key_users": "Collahuasi, Quebrada Blanca"},
    {"name": "Barquito",          "region": "Atacama",     "lat": -27.07, "lon": -70.84,
     "products": "Cu concentrate, cathode", "key_users": "Codelco Salvador, ENAMI"},
    {"name": "Caldera",           "region": "Atacama",     "lat": -27.07, "lon": -70.82,
     "products": "Cu concentrate", "key_users": "Medium miners"},
    {"name": "Coquimbo",          "region": "Coquimbo",    "lat": -29.96, "lon": -71.35,
     "products": "Cu concentrate, Fe pellets", "key_users": "Los Pelambres, CMP"},
    {"name": "Ventanas",          "region": "Valparaiso",  "lat": -32.74, "lon": -71.49,
     "products": "Cu cathode, blister", "key_users": "Codelco, Anglo American"},
    {"name": "San Antonio",       "region": "Valparaiso",  "lat": -33.59, "lon": -71.62,
     "products": "Cu cathode, general", "key_users": "El Teniente, general"},
]

SMELTER_NAME_MAP = {
    "Chuquicamata smelter":        "Chuquicamata SX-EW plant (oxide) and smelter",
    "Potrerillos smelter":         "Potrerillos SX-EW refinery and smelter",
    "Caletones smelter":           "Caletones smelter (anodes). refinery (fire-refined ingots), and SX-EW plant",
    "Altonorte smelter":           "Altonorte smelter",
    "Paipote smelter (H.V. Lira)": "Hernán Videla Lira smelter (anodes and blister)",
    "Chagres smelter":             "Chagres smelter (anodes and blister)",
}

# ── Utility functions ──────────────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    """Distance in km between two coordinate pairs."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def parse_comm_list(val):
    if pd.isna(val):
        return []
    return [x.strip() for x in str(val).split(",") if x.strip()]

def add_commodity(row_idx, commodity, df, col):
    current = parse_comm_list(df.at[row_idx, col])
    if commodity not in current:
        current.append(commodity)
        df.at[row_idx, col] = ", ".join(current)
        return True
    return False

def nearest_port(lat, lon, product_type="concentrate"):
    best_dist, best_port = float("inf"), None
    for port in PORTS:
        if product_type == "cathode" and "cathode" not in port["products"].lower():
            continue
        if product_type == "concentrate" and "concentrate" not in port["products"].lower():
            continue
        dist = haversine_km(lat, lon, port["lat"], port["lon"])
        if dist < best_dist:
            best_dist, best_port = dist, port
    return best_port, best_dist

def section_header(title, width=65):
    print(f"\n{'=' * width}")
    print(title)
    print("=" * width)

# ── Load data (once) ───────────────────────────────────────────────────────

inv_path = os.path.join(DIR_PRELIM, "Chile_Minerals_Inventory.csv")
links_path = os.path.join(DIR_PRELIM, "Chile_Mine_Plant_Links.csv")

inv = pd.read_csv(inv_path, low_memory=False)
links = pd.read_csv(links_path)

# Back up originals
for p in [inv_path, links_path]:
    bak = p.replace(".csv", "_backup.csv")
    if not os.path.exists(bak):
        shutil.copy2(p, bak)
        print(f"Backed up: {bak}")

comm_col = "COMMODITY_LIST_STR" if "COMMODITY_LIST_STR" in inv.columns else "ALL_COMMODITIES_RAW"

print(f"Inventory: {len(inv)} records")
print(f"Links: {len(links)} records")

# ── Remove idle mines from supply chain edges ─────────────────────────────
# Idle mines have no throughput and generate phantom edges via distance matching.
# Keep them in inventory (for latent capacity context) but remove their links.

idle_mines = set(inv.loc[inv["FACILITY_TYPE"] == "Mine (idle)", "FACILITY_NAME"])
idle_link_mask = links["MINE_NAME"].isin(idle_mines)
n_idle_links = idle_link_mask.sum()
links = links[~idle_link_mask].reset_index(drop=True)
print(f"Removed {n_idle_links} links from {len(idle_mines)} idle mines (kept in inventory)")
print(f"Links after idle filter: {len(links)}")


# %% 2. COCHILCO INTEGRATION (Cu, Mo, Li)

# ── 2A. Codelco division mapping ──────────────────────────────────────────

section_header("2A. CODELCO DIVISION MAPPING")

inv["_name_lower"] = inv["FACILITY_NAME"].str.lower().str.strip()
codelco_matches = {}

for div_name, info in CODELCO_MAP.items():
    matched_idx = []
    for term in info["search"]:
        mask = inv["_name_lower"].str.contains(term.lower(), na=False, regex=False)
        matched_idx.extend(inv[mask].index.tolist())
    matched_idx = list(set(matched_idx))
    codelco_matches[div_name] = matched_idx

    if matched_idx:
        print(f"  {div_name}")
        for idx in matched_idx:
            if pd.isna(inv.at[idx, "OPERATOR_NAME"]) or inv.at[idx, "OPERATOR_NAME"] == "":
                inv.at[idx, "OPERATOR_NAME"] = "Codelco"
            print(f"    -> {inv.at[idx, 'FACILITY_NAME']} ({inv.at[idx, 'FACILITY_TYPE']})")
    else:
        print(f"  {div_name} -> NO MATCH")

inv.drop(columns=["_name_lower"], inplace=True)

# ── 2B. Copper production matching ────────────────────────────────────────

section_header("2B. COPPER PRODUCTION MATCHING")

nat = pd.read_excel(COCHILCO_PATH, sheet_name="A_National_Production", header=3, index_col=0)
nat.columns = [int(c) if isinstance(c, (int, float)) else c for c in nat.columns]
latest_yr = max(c for c in nat.columns if isinstance(c, int))

commodity_search = {
    "Copper": "COBRE.*Miles de TM", "Molybdenum": "MOLIBDENO.*TM de fino",
    "Gold": "ORO.*Kg de fino", "Silver": "PLATA.*Kg de fino",
    "Iron": "HIERRO.*Miles de TM", "Zinc": "ZINC.*TM de fino",
}
print(f"COCHILCO latest year: {latest_yr}")
for comm, pattern in commodity_search.items():
    match = [r for r in nat.index if re.search(pattern, str(r), re.IGNORECASE)]
    if match:
        print(f"  {comm:<15} {nat.loc[match[0], latest_yr]:>15,.1f}")

cu_co = pd.read_excel(COCHILCO_PATH, sheet_name="B1_Copper_by_Company", header=3, index_col=0)
cu_co.columns = [int(c) if isinstance(c, (int, float)) else c for c in cu_co.columns]

inv["COCHILCO_CU_2024_KMT"] = np.nan
inv["COCHILCO_COMPANY"] = ""
cu_matched, cu_matched_prod = 0, 0.0

for company, search_terms in COMPANY_TO_DEPOSIT.items():
    if company not in cu_co.index or latest_yr not in cu_co.columns:
        continue
    prod = cu_co.loc[company, latest_yr]
    if not isinstance(prod, (int, float)) or pd.isna(prod):
        continue

    for term in search_terms:
        mask = inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False)
        mine_mask = mask & inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matches = inv[mine_mask] if mine_mask.any() else inv[mask]
        if len(matches) > 0:
            idx = matches.index[0]
            if pd.isna(inv.at[idx, "COCHILCO_CU_2024_KMT"]):
                inv.at[idx, "COCHILCO_CU_2024_KMT"] = prod
                inv.at[idx, "COCHILCO_COMPANY"] = company
                cu_matched += 1
                cu_matched_prod += prod
                print(f"  {company:<35} -> {inv.at[idx, 'FACILITY_NAME']:<35} {prod:>8,.1f} kMT")
            break
    else:
        print(f"  {company:<35} -> NOT MATCHED ({prod:,.1f} kMT)")

cu_total = cu_co.loc["TOTAL", latest_yr] if "TOTAL" in cu_co.index else 5506.0
print(f"\nCu matched: {cu_matched} companies, {cu_matched_prod:,.1f} / {cu_total:,.1f} kMT "
      f"({cu_matched_prod/cu_total*100:.1f}%)")

# ── 2C. Molybdenum production ─────────────────────────────────────────────

section_header("2C. MOLYBDENUM PRODUCTION MATCHING")

inv["COCHILCO_MO_2024_MT"] = np.nan

if os.path.exists(COCHILCO_ORIG):
    wb_orig = openpyxl.load_workbook(COCHILCO_ORIG, read_only=True, data_only=True)
    ws = wb_orig["Tabla 4.2"]
    rows_raw = list(ws.iter_rows(values_only=True))

    yr_row_idx, year_cols = None, {}
    for i, row in enumerate(rows_raw):
        if sum(1 for v in row if isinstance(v, (int, float)) and 2014 < v < 2025) >= 8:
            yr_row_idx = i
            for j, v in enumerate(row):
                if isinstance(v, (int, float)) and 2014 < v < 2025:
                    year_cols[int(v)] = j
            break

    mo_by_company = {}
    current_company = None
    for i in range(yr_row_idx + 1, len(rows_raw)):
        row = rows_raw[i]
        label = str(row[0]).strip() if row[0] is not None else ""
        if not label or label.startswith("(") or label.startswith("Fuente"):
            continue

        has_data = any(isinstance(row[j], (int, float)) and row[j] != 0
                       for j in year_cols.values() if j < len(row))

        if has_data and "CONCENTRADO" not in label and "ÓXIDO" not in label:
            vals = {yr: row[col] for yr, col in year_cols.items()
                    if col < len(row) and isinstance(row[col], (int, float))}
            mo_by_company[label] = vals
            current_company = None
        elif not has_data and "CONCENTRADO" not in label and "ÓXIDO" not in label:
            current_company = label
        elif has_data and current_company:
            vals = {yr: row[col] for yr, col in year_cols.items()
                    if col < len(row) and isinstance(row[col], (int, float))}
            if current_company not in mo_by_company:
                mo_by_company[current_company] = {}
            for yr, v in vals.items():
                mo_by_company[current_company][yr] = mo_by_company[current_company].get(yr, 0) + v

    MO_MAP = {
        "Divisiones Chuquicamata y Radomiro Tomic": ["Chuquicamata", "Radomiro Tomic"],
        "División Salvador": ["Salvador"], "División Andina": ["Andina"],
        "División El Teniente": ["El Teniente", "Teniente"],
        "Collahuasi": ["Collahuasi"], "Sierra Gorda": ["Sierra Gorda"],
        "Caserones": ["Caserones"], "Los Pelambres": ["Los Pelambres", "Pelambres"],
        "Anglo American Sur": ["Los Bronces", "Bronces"],
        "Centinela": ["Centinela"], "Spence": ["Spence"],
        "Quebrada Blanca": ["Quebrada Blanca"],
    }

    mo_matched, mo_matched_prod = 0, 0.0
    for company, search_terms in MO_MAP.items():
        if company not in mo_by_company:
            continue
        prod = mo_by_company[company].get(2024, 0)
        if prod <= 0:
            continue
        for term in search_terms:
            mask = (inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False) &
                    inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False))
            matches = inv[mask]
            if len(matches) > 0:
                idx = matches.index[0]
                if pd.isna(inv.at[idx, "COCHILCO_MO_2024_MT"]):
                    inv.at[idx, "COCHILCO_MO_2024_MT"] = prod
                    mo_matched += 1
                    mo_matched_prod += prod
                    print(f"  {company:<50} -> {inv.at[idx, 'FACILITY_NAME']:<30} {prod:>10,.1f} MT")
                break
        else:
            print(f"  {company:<50} -> NOT MATCHED ({prod:,.1f} MT)")

    mo_national = mo_by_company.get("TOTAL", {}).get(2024, 38487)
    print(f"\nMo matched: {mo_matched}, {mo_matched_prod:,.1f} / {mo_national:,.1f} MT "
          f"({mo_matched_prod/mo_national*100:.1f}%)")
    wb_orig.close()
else:
    print(f"  Original COCHILCO not found at {COCHILCO_ORIG}, skipping Mo parsing")

# ── 2D. Lithium fixes + link cleanup ─────────────────────────────────────

section_header("2D. LITHIUM FIXES AND LINK CLEANUP")

potash_li = inv[
    (inv["SOURCE"].str.contains("USGS", na=False)) &
    (inv["PRIMARY_COMMODITY"].str.contains("Potash", case=False, na=False)) &
    (inv["FACILITY_NAME"].str.contains("Salar|Carmen|Atacama", case=False, na=False))
]
li_fixes = 0
for idx, row in potash_li.iterrows():
    if add_commodity(idx, "Lithium", inv, comm_col):
        li_fixes += 1
        print(f"  Added Lithium to: {row['FACILITY_NAME']}")

salar_mine = inv[(inv["FACILITY_NAME"] == "Salar de Atacama") &
                 (inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False))]
for idx in salar_mine.index:
    add_commodity(idx, "Potassium", inv, comm_col)

print(f"  Lithium commodity fixes: {li_fixes} records")

# Rebuild lithium links
FACILITY_STAGE_LI = {
    "Mine (active)": "extraction", "Mine (idle)": "extraction_idle",
    "Prospect/Project": "extraction", "Mine (USGS)": "extraction",
    "Concentrator": "processing", "SX-EW Plant": "processing",
    "Smelter": "processing", "Refinery": "processing",
    "Processing Plant": "processing", "Pellet Plant": "processing",
    "Grinding Plant": "processing", "Steel Plant": "processing",
}

inv["_stage"] = inv["FACILITY_TYPE"].map(FACILITY_STAGE_LI)

li_mines = inv[(inv["_stage"] == "extraction") &
               inv[comm_col].str.contains("Lithium", case=False, na=False) &
               inv["LATITUD"].notna()]
li_plants = inv[(inv["_stage"] == "processing") &
                inv[comm_col].str.contains("Lithium", case=False, na=False) &
                inv["LATITUD"].notna()]

new_li_links = []
for _, mrow in li_mines.iterrows():
    for _, prow in li_plants.iterrows():
        dist = haversine_km(mrow["LATITUD"], mrow["LONGITUD"], prow["LATITUD"], prow["LONGITUD"])
        if dist <= 300:
            new_li_links.append({
                "MINE_NAME": mrow["FACILITY_NAME"], "MINE_TYPE": mrow["FACILITY_TYPE"],
                "MINE_STATUS": mrow.get("STATUS", ""), "MINE_LAT": mrow["LATITUD"],
                "MINE_LON": mrow["LONGITUD"], "MINE_REGION": mrow.get("REGION", ""),
                "MINE_OPERATOR": mrow.get("OPERATOR_NAME", ""),
                "PLANT_NAME": prow["FACILITY_NAME"], "PLANT_TYPE": prow["FACILITY_TYPE"],
                "PLANT_STATUS": prow.get("STATUS", ""), "PLANT_LAT": prow["LATITUD"],
                "PLANT_LON": prow["LONGITUD"], "PLANT_OPERATOR": prow.get("OPERATOR_NAME", ""),
                "PLANT_OWNER": prow.get("OWNER_NAME", ""),
                "PLANT_CAPACITY": prow.get("CAPACITY"),
                "PLANT_CAPACITY_UNITS": prow.get("CAPACITY_UNITS", ""),
                "SHARED_COMMODITIES": "Lithium", "DISTANCE_KM": round(dist, 1),
            })

old_li = len(links[links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)])
links = links[~links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)]

if new_li_links:
    li_df = pd.DataFrame(new_li_links)
    for col in links.columns:
        if col not in li_df.columns:
            li_df[col] = np.nan
    links = pd.concat([links, li_df[links.columns]], ignore_index=True)

li_mask = links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)
prospect_far = li_mask & links["MINE_TYPE"].str.contains("Prospect", case=False, na=False) & (links["DISTANCE_KM"] > 200)
links = links[~prospect_far]

li_subset = links[links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)].copy()
non_li = links[~links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)]
li_subset["_loc"] = li_subset["PLANT_LAT"].round(2).astype(str) + "_" + li_subset["PLANT_LON"].round(2).astype(str)
li_deduped = li_subset.sort_values("DISTANCE_KM").drop_duplicates(subset=["MINE_NAME", "_loc"], keep="first")
li_deduped = li_deduped.drop(columns=["_loc"])
links = pd.concat([non_li, li_deduped], ignore_index=True)
links.sort_values(["MINE_NAME", "DISTANCE_KM"], inplace=True)

inv.drop(columns=["_stage"], inplace=True, errors="ignore")

li_final = len(links[links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)])
print(f"  Lithium links: {old_li} old -> {li_final} cleaned")


# %% 3. DOWNSTREAM SUPPLY CHAIN

# ── 3A. Processing stage classification ───────────────────────────────────

section_header("3A. PROCESSING STAGE CLASSIFICATION")

STAGE_MAP = {
    "Mine (active)": "extraction", "Mine (idle)": "extraction_idle",
    "Mine (USGS)": "extraction", "Prospect/Project": "extraction",
    "Concentrator": "concentration", "SX-EW Plant": "sx_ew",
    "Smelter": "smelting", "Refinery": "refining",
    "Processing Plant": "processing", "Pellet Plant": "processing",
    "Grinding Plant": "processing", "Steel Plant": "processing",
}

inv["CHAIN_STAGE"] = inv["FACILITY_TYPE"].map(STAGE_MAP).fillna("other")

# Refine "processing" based on facility name keywords
proc_mask = inv["CHAIN_STAGE"] == "processing"
for idx in inv[proc_mask].index:
    name = str(inv.at[idx, "FACILITY_NAME"]).lower()
    if any(kw in name for kw in ["smelter", "fundici", "smelting"]):
        inv.at[idx, "CHAIN_STAGE"] = "smelting"
    elif any(kw in name for kw in ["refin", "electro"]):
        inv.at[idx, "CHAIN_STAGE"] = "refining"
    elif any(kw in name for kw in ["sx-ew", "sx/ew", "leach", "lixiv", "cathode", "catod"]):
        inv.at[idx, "CHAIN_STAGE"] = "sx_ew"
    elif any(kw in name for kw in ["concentrat", "flotation", "mill"]):
        inv.at[idx, "CHAIN_STAGE"] = "concentration"

for stage, count in inv["CHAIN_STAGE"].value_counts().items():
    print(f"  {stage:<18} {count:>4}")

# ── 3B. Smelter matching to inventory ─────────────────────────────────────

section_header("3B. SMELTER AND PORT SETUP")

smelter_inv_map = {}
for sm in SMELTERS:
    for term in sm["search"]:
        mask = inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False)
        hits = inv[mask]
        if len(hits) > 0:
            idx = hits.index[0]
            smelter_inv_map[sm["name"]] = inv.at[idx, "FACILITY_NAME"]
            inv.at[idx, "CHAIN_STAGE"] = "smelting"
            print(f"  {sm['name']:<30} -> {inv.at[idx, 'FACILITY_NAME']}")
            break
    else:
        print(f"  {sm['name']:<30} -> NOT IN INVENTORY")

ports_df = pd.DataFrame(PORTS)
ports_df.to_csv(os.path.join(DIR_PRELIM, "Chile_Ports.csv"), index=False)
print(f"\n  Ports saved: {len(ports_df)}")

# ── 3C. Product form assignment ───────────────────────────────────────────

section_header("3C. PRODUCT FORM + DOWNSTREAM EDGES")

links["PRODUCT_FORM"] = "unknown"
for idx, row in links.iterrows():
    pt = str(row.get("PLANT_TYPE", "")).lower()
    pn = str(row.get("PLANT_NAME", "")).lower()
    if any(kw in pt for kw in ["sx-ew", "sx/ew"]) or any(kw in pn for kw in ["sx-ew", "sx/ew", "leach", "cathode"]):
        links.at[idx, "PRODUCT_FORM"] = "cathode_sxew"
    elif any(kw in pt for kw in ["smelter"]) or any(kw in pn for kw in ["smelter", "fundici"]):
        links.at[idx, "PRODUCT_FORM"] = "blister"
    elif any(kw in pt for kw in ["refin"]) or any(kw in pn for kw in ["refin", "electro"]):
        links.at[idx, "PRODUCT_FORM"] = "cathode_er"
    elif any(kw in pt for kw in ["concentrat"]) or any(kw in pn for kw in ["concentrat", "flotation", "mill"]):
        links.at[idx, "PRODUCT_FORM"] = "concentrate"
    else:
        links.at[idx, "PRODUCT_FORM"] = "concentrate"

print("Product forms in mine-plant links:")
print(links["PRODUCT_FORM"].value_counts().to_string())

# ── 3D. Build downstream edges ────────────────────────────────────────────

downstream_edges = []
concentrators = inv[inv["CHAIN_STAGE"] == "concentration"].copy()

# A. Concentrator -> Smelter (integrated, via named feed mines)
smelter_fed = set()
for sm in SMELTERS:
    # Use inventory name if we resolved it, otherwise canonical name
    sm_inv_name = smelter_inv_map.get(sm["name"], sm["name"])
    for mine_term in sm.get("feeds_from_mines", []):
        concs = concentrators[concentrators["FACILITY_NAME"].str.contains(
            mine_term, case=False, na=False, regex=False)]
        for cidx, crow in concs.iterrows():
            downstream_edges.append({
                "FROM_NAME": crow["FACILITY_NAME"], "FROM_TYPE": "concentrator",
                "FROM_LAT": crow["LATITUD"], "FROM_LON": crow["LONGITUD"],
                "TO_NAME": sm_inv_name, "TO_TYPE": "smelter",
                "TO_LAT": sm["lat"], "TO_LON": sm["lon"],
                "EDGE_TYPE": "concentrate_to_smelter", "PRODUCT_FORM": "concentrate",
                "OPERATOR": sm["operator"],
                "DISTANCE_KM": haversine_km(crow["LATITUD"], crow["LONGITUD"], sm["lat"], sm["lon"])
                if pd.notna(crow["LATITUD"]) else None,
            })
            smelter_fed.add(cidx)

# B. Smelter -> Port
for sm in SMELTERS:
    sm_inv_name = smelter_inv_map.get(sm["name"], sm["name"])
    for port_name in sm["export_ports"]:
        port = next((p for p in PORTS if p["name"] == port_name), None)
        if port:
            downstream_edges.append({
                "FROM_NAME": sm_inv_name, "FROM_TYPE": "smelter",
                "FROM_LAT": sm["lat"], "FROM_LON": sm["lon"],
                "TO_NAME": port["name"], "TO_TYPE": "port",
                "TO_LAT": port["lat"], "TO_LON": port["lon"],
                "EDGE_TYPE": "smelter_to_port", "PRODUCT_FORM": sm["output_product"],
                "OPERATOR": sm["operator"],
                "DISTANCE_KM": haversine_km(sm["lat"], sm["lon"], port["lat"], port["lon"]),
            })

# C. Concentrator -> Port (concentrate export, for non-smelter-fed concentrators)
# Also track smelter-fed from mine term matching
for sm in SMELTERS:
    for mt in sm.get("feeds_from_mines", []):
        smelter_fed.update(concentrators[concentrators["FACILITY_NAME"].str.contains(
            mt, case=False, na=False, regex=False)].index)

# Escondida -> Coloso (hardcoded)
esc = concentrators[concentrators["FACILITY_NAME"].str.contains("Escondida", case=False, na=False, regex=False)]
coloso = next(p for p in PORTS if p["name"] == "Coloso")
for cidx, crow in esc.iterrows():
    downstream_edges.append({
        "FROM_NAME": crow["FACILITY_NAME"], "FROM_TYPE": "concentrator",
        "FROM_LAT": crow["LATITUD"], "FROM_LON": crow["LONGITUD"],
        "TO_NAME": "Coloso", "TO_TYPE": "port",
        "TO_LAT": coloso["lat"], "TO_LON": coloso["lon"],
        "EDGE_TYPE": "concentrate_to_port", "PRODUCT_FORM": "concentrate",
        "OPERATOR": crow.get("OPERATOR_NAME", ""),
        "DISTANCE_KM": haversine_km(crow["LATITUD"], crow["LONGITUD"], coloso["lat"], coloso["lon"])
        if pd.notna(crow["LATITUD"]) else None,
    })
    smelter_fed.add(cidx)

# Remaining concentrators -> nearest port
for cidx, crow in concentrators.iterrows():
    if cidx in smelter_fed or pd.isna(crow["LATITUD"]):
        continue
    port, dist = nearest_port(crow["LATITUD"], crow["LONGITUD"], "concentrate")
    if port:
        downstream_edges.append({
            "FROM_NAME": crow["FACILITY_NAME"], "FROM_TYPE": "concentrator",
            "FROM_LAT": crow["LATITUD"], "FROM_LON": crow["LONGITUD"],
            "TO_NAME": port["name"], "TO_TYPE": "port",
            "TO_LAT": port["lat"], "TO_LON": port["lon"],
            "EDGE_TYPE": "concentrate_to_port", "PRODUCT_FORM": "concentrate",
            "OPERATOR": crow.get("OPERATOR_NAME", ""),
            "DISTANCE_KM": round(dist, 1),
        })

# D. SX-EW -> Port
sxew = inv[inv["CHAIN_STAGE"] == "sx_ew"].copy()
for pidx, prow in sxew.iterrows():
    if pd.isna(prow["LATITUD"]):
        continue
    port, dist = nearest_port(prow["LATITUD"], prow["LONGITUD"], "cathode")
    if port:
        downstream_edges.append({
            "FROM_NAME": prow["FACILITY_NAME"], "FROM_TYPE": "sx_ew",
            "FROM_LAT": prow["LATITUD"], "FROM_LON": prow["LONGITUD"],
            "TO_NAME": port["name"], "TO_TYPE": "port",
            "TO_LAT": port["lat"], "TO_LON": port["lon"],
            "EDGE_TYPE": "sxew_to_port", "PRODUCT_FORM": "cathode_sxew",
            "OPERATOR": prow.get("OPERATOR_NAME", ""),
            "DISTANCE_KM": round(dist, 1),
        })

for etype in ["concentrate_to_smelter", "smelter_to_port", "concentrate_to_port", "sxew_to_port"]:
    n = sum(1 for e in downstream_edges if e["EDGE_TYPE"] == etype)
    print(f"  {etype:<25} {n:>4}")

# ── 3E. Unified edge table ────────────────────────────────────────────────

section_header("3E. UNIFIED SUPPLY CHAIN EDGES")

upstream = links[["MINE_NAME", "PLANT_NAME", "MINE_LAT", "MINE_LON",
                  "PLANT_LAT", "PLANT_LON", "SHARED_COMMODITIES",
                  "DISTANCE_KM", "PRODUCT_FORM"]].copy()
upstream.rename(columns={
    "MINE_NAME": "FROM_NAME", "PLANT_NAME": "TO_NAME",
    "MINE_LAT": "FROM_LAT", "MINE_LON": "FROM_LON",
    "PLANT_LAT": "TO_LAT", "PLANT_LON": "TO_LON",
    "SHARED_COMMODITIES": "COMMODITIES",
}, inplace=True)
upstream["FROM_TYPE"] = "mine"
upstream["TO_TYPE"] = "plant"
upstream["EDGE_TYPE"] = "mine_to_plant"

downstream_df = pd.DataFrame(downstream_edges)
downstream_df["COMMODITIES"] = "Copper"
downstream_df["DISTANCE_KM"] = downstream_df["DISTANCE_KM"].round(1)

common_cols = ["FROM_NAME", "FROM_TYPE", "FROM_LAT", "FROM_LON",
               "TO_NAME", "TO_TYPE", "TO_LAT", "TO_LON",
               "EDGE_TYPE", "PRODUCT_FORM", "COMMODITIES", "DISTANCE_KM"]
for col in common_cols:
    if col not in upstream.columns:
        upstream[col] = ""
    if col not in downstream_df.columns:
        downstream_df[col] = ""

edges = pd.concat([upstream[common_cols], downstream_df[common_cols]], ignore_index=True)

print(f"Unified edges: {len(edges)}")
for et, count in edges["EDGE_TYPE"].value_counts().items():
    print(f"  {et:<25} {count:>5}")


# %% 4. EXPORT DESTINATIONS (COCHILCO Section D)

section_header("4. EXPORT DESTINATIONS")

wb = openpyxl.load_workbook(COCHILCO_ORIG, read_only=True, data_only=True)

def parse_destination_table(sheet_name, target_year=2024):
    """Extract country -> value dict for a given year from a COCHILCO destination table."""
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))

    yr_row, yc = None, None
    for i, row in enumerate(rows):
        if sum(1 for v in row if isinstance(v, (int, float)) and 2014 < v < 2025) >= 5:
            yr_row = i
            for j, v in enumerate(row):
                if isinstance(v, (int, float)) and int(v) == target_year:
                    yc = j
            break

    if yr_row is None or yc is None:
        print(f"  WARNING: Could not find {target_year} in {sheet_name}")
        return {}

    result = {}
    current_region = None
    for i in range(yr_row + 1, len(rows)):
        row = rows[i]
        label = str(row[0]).strip() if row[0] is not None else ""
        val = row[yc] if yc < len(row) else None

        if not label or label.startswith("(") or label.startswith("Fuente") or label.startswith("Source"):
            continue
        if "EUROPA" in label or "AMÉRICA" in label or "ASIA" in label or "OCEANÍA" in label or "AFRICA" in label:
            current_region = label.split("/")[0].strip()
            continue
        if label in ("TOTAL", "OTROS / Other"):
            continue

        if isinstance(val, (int, float)) and val > 0:
            parts = label.split("/")
            name_en = parts[-1].strip() if len(parts) > 1 else parts[0].strip()
            name_es = parts[0].strip()
            country = name_en if name_en else name_es
            result[country] = {"value": val, "region": current_region or ""}

    return result

def parse_nonmetallic_column(sheet_name, col_idx, target_rows=(10, 65)):
    """Parse a specific column from Tabla 11 (non-metallic exports, value in $M FOB)."""
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    result = {}
    current_region = None

    for i in range(target_rows[0], min(target_rows[1], len(rows))):
        row = rows[i]
        label = str(row[0]).strip() if row[0] is not None else ""
        val = row[col_idx] if col_idx < len(row) else None

        if not label:
            continue
        if "EUROPA" in label or "AMÉRICA" in label or "ASIA" in label:
            current_region = label.split("/")[0].strip()
            continue
        if label in ("TOTAL", "TOTAL MINERÍA"):
            continue

        if isinstance(val, (int, float)) and val > 0:
            parts = label.split("/")
            country = parts[-1].strip() if len(parts) > 1 else parts[0].strip()
            result[country] = {"value": val, "region": current_region or ""}

    return result

print("Parsing export destination tables...\n")

cu_refined = parse_destination_table("Tabla 18.2")
cu_blister = parse_destination_table("Tabla 19.2")
cu_concentrate = parse_destination_table("Tabla 20.2")
mo_concentrate = parse_destination_table("Tabla 23.2")
li_exports = parse_nonmetallic_column("Tabla 11", col_idx=2)
io_exports = parse_nonmetallic_column("Tabla 11", col_idx=7)

for name, data, unit in [
    ("Cu refined", cu_refined, "kMT"), ("Cu blister", cu_blister, "kMT"),
    ("Cu concentrate", cu_concentrate, "kMT"), ("Mo concentrate", mo_concentrate, "MT"),
    ("Lithium", li_exports, "$M FOB"), ("Iodine", io_exports, "$M FOB"),
]:
    total = sum(d["value"] for d in data.values())
    print(f"  {name:<20} {len(data):>3} destinations, total: {total:>10,.1f} {unit}")

wb.close()

# Country coordinates and aliases
COUNTRY_COORDS = {
    "China": {"lat": 31.23, "lon": 121.47}, "Japan": {"lat": 35.68, "lon": 139.69},
    "South Korea": {"lat": 37.57, "lon": 126.98}, "USA": {"lat": 29.76, "lon": -95.37},
    "Brazil": {"lat": -23.55, "lon": -46.63}, "India": {"lat": 19.08, "lon": 72.88},
    "Germany": {"lat": 53.55, "lon": 9.99}, "Spain": {"lat": 36.72, "lon": -4.42},
    "France": {"lat": 48.86, "lon": 2.35}, "Italy": {"lat": 45.46, "lon": 9.19},
    "Netherlands": {"lat": 51.92, "lon": 4.48}, "Belgium": {"lat": 51.26, "lon": 4.35},
    "Sweden": {"lat": 57.71, "lon": 11.97}, "Bulgaria": {"lat": 42.70, "lon": 23.32},
    "Finland": {"lat": 60.17, "lon": 24.94}, "Canada": {"lat": 49.28, "lon": -123.12},
    "Mexico": {"lat": 19.43, "lon": -99.13}, "Taiwan": {"lat": 25.03, "lon": 121.57},
    "Thailand": {"lat": 13.76, "lon": 100.50}, "Philippines": {"lat": 14.60, "lon": 120.98},
    "Malaysia": {"lat": 3.14, "lon": 101.69}, "Indonesia": {"lat": -6.21, "lon": 106.85},
    "Vietnam": {"lat": 10.82, "lon": 106.63}, "Peru": {"lat": -12.05, "lon": -77.04},
    "Colombia": {"lat": 4.71, "lon": -74.07}, "Argentina": {"lat": -34.60, "lon": -58.38},
    "Turkey": {"lat": 41.01, "lon": 28.98}, "United Kingdom": {"lat": 51.51, "lon": -0.13},
    "Switzerland": {"lat": 47.38, "lon": 8.54}, "Singapore": {"lat": 1.35, "lon": 103.82},
    "Greece": {"lat": 37.98, "lon": 23.73}, "Portugal": {"lat": 38.72, "lon": -9.14},
    "Panama": {"lat": 8.98, "lon": -79.52}, "Bahrain": {"lat": 26.07, "lon": 50.56},
    "UAE": {"lat": 25.20, "lon": 55.27}, "Hong Kong": {"lat": 22.32, "lon": 114.17},
    "Poland": {"lat": 52.23, "lon": 21.01}, "Norway": {"lat": 59.91, "lon": 10.75},
}

COUNTRY_ALIAS = {
    "Corea del Sur": "South Korea", "Estados Unidos": "USA",
    "Emiratos Árabes Unidos": "UAE", "United Arab Emirates": "UAE",
    "Baréin": "Bahrain", "Turquía": "Turkey", "Taiwán": "Taiwan",
    "Tailandia": "Thailand", "Filipinas": "Philippines",
    "Malasia": "Malaysia", "Singapur": "Singapore",
    "Panamá": "Panama", "Noruega": "Norway",
}

def normalize_country(name):
    return COUNTRY_ALIAS.get(name, name)

# ── 4A. ADUANAS-DERIVED PORT SHARES ──────────────────────────────────────
# Replaces hardcoded PORT_PRODUCT_MAP with actual shares from Aduanas Salidas
# data. Also extracts export flows for commodities not in COCHILCO tables
# (gold, iron, silver, manganese, boron, rhenium, nitrate).
#
# Falls back to manual estimates if Salidas2025.csv is not found.

_salidas_candidates = [
    os.path.join(BASE_DIR, "data", "Salidas2024.csv"),
    os.path.join(BASE_DIR, "data", "Salidas2025.csv"),
]
SALIDAS_PATH = next((p for p in _salidas_candidates if os.path.exists(p)), _salidas_candidates[0])
CODIGOS_PATH = os.path.join(BASE_DIR, "data", "tablas_de_codigos.xlsx")

# Aduanas export operation codes (filter out re-exports, temporary, etc.)
EXPORT_OP_CODES = {"200", "201", "202", "203", "204", "205", "206", "207",
                   "210", "211", "212", "213", "216"}

# HS 4-digit (or 6/7-digit where needed) -> (commodity, product_form)
HS_MINERAL_MAP = {
    "2603": ("Copper", "concentrate"),
    "7402": ("Copper", "blister"),
    "7403": ("Copper", "cathode"),
    "2613": ("Molybdenum", "mo_concentrate"),
    "2601": ("Iron", "iron_ore"),
    "7108": ("Gold", "gold_refined"),
    "7106": ("Silver", "silver_refined"),
    "283620": ("Lithium", "lithium_compounds"),    # lithium carbonate (2836.20xx)
    "28362010": ("Lithium", "lithium_compounds"),  # lithium carbonate grade 1
    "28362020": ("Lithium", "lithium_compounds"),  # lithium carbonate grade 2
    "28362030": ("Lithium", "lithium_compounds"),  # lithium carbonate other
    "283691": ("Lithium", "lithium_compounds"),    # other lithium carbonates
    "28369100": ("Lithium", "lithium_compounds"),  # 8-digit
    "282520": ("Lithium", "lithium_compounds"),    # lithium hydroxide
    "28252000": ("Lithium", "lithium_compounds"),  # 8-digit lithium hydroxide
    "284290": ("Lithium", "lithium_compounds"),    # other lithium salts
    "280120": ("Iodine", "iodine"),
    "2528": ("Boron", "borate"),
    "2602": ("Manganese", "mn_ore"),
    "2834": ("Nitrate", "nitrate"),
    "284170": ("Rhenium", "perrhenate"),
}

# Aduanas port code -> pipeline port name (manual aliases for pipeline name matching)
# These map codigos NOMBRE_PUERTO -> pipeline PORTS[].name where names differ
ADUANAS_PORT_ALIAS = {
    "CALETA COLOSO": "Coloso", "PUERTO ANGAMOS": "Angamos",
    "ANTOFAGASTA": "Antofagasta (ATI)", "CHAÑARAL / BARQUITO": "Barquito",
    "HUASCO / GUACOLDA": "Huasco", "GUAYACÁN": "Guayacán",
    "CAP. HUACHIPATO": "Huachipato",
    "AEROP. A.M. BENITEZ": "Santiago (air)",    # Arturo Merino Benítez / SCL
    "AEROP. CERRO MORENO": "Antofagasta (air)",
    "AEROP. CHACALLUTA": "Arica (air)",
    "AEROP. DIEGO ARACENA": "Iquique (air)",
    "AEROP. EL TEPUAL": "Puerto Montt (air)",
    "AEROP. C.I. DEL CAMPO": "Santiago (air)",  # alternate name
    "CHACABUCO / PUERTO AYSÉN": "Puerto Aysén",
    "TERMINAL PETROLERO ENAP": "ENAP terminal",
    "OTROS PUERTOS CHILENOS": "Other",
}
# Codes not in codigos file (legacy/internal)
ADUANAS_MANUAL_PORTS = {
    827: "Unknown (827)",
}

def parse_codigos_sheet(path, sheet_name, key_col_name, value_col_name):
    """Parse a codigos lookup sheet into {int_code: str_value} dict.
    Finds column headers by exact match, then reads rows below."""
    _xl = pd.read_excel(path, sheet_name=sheet_name, header=None)
    key_idx, val_idx, header_row = None, None, None
    for i in range(min(15, len(_xl))):
        for j, v in enumerate(_xl.iloc[i].values):
            if pd.notna(v) and str(v).strip().upper() == key_col_name.upper():
                key_idx = j
                header_row = i
                break
        if header_row is not None:
            break
    if header_row is None:
        return {}
    for j, v in enumerate(_xl.iloc[header_row].values):
        if pd.notna(v) and value_col_name.upper() in str(v).strip().upper():
            val_idx = j
            break
    if val_idx is None:
        return {}
    result = {}
    for ii in range(header_row + 1, len(_xl)):
        k = _xl.iloc[ii, key_idx]
        v = _xl.iloc[ii, val_idx]
        if pd.isna(k):
            continue
        try:
            result[int(float(k))] = str(v).strip()
        except (ValueError, TypeError):
            pass
    return result

# Fallback if Aduanas data unavailable
PORT_PRODUCT_MAP_FALLBACK = {
    "concentrate": {
        "Coloso": 0.35, "Mejillones": 0.20, "Patache": 0.15,
        "Barquito": 0.08, "Coquimbo": 0.12, "Angamos": 0.05, "Caldera": 0.05,
    },
    "cathode": {
        "Angamos": 0.25, "Mejillones": 0.15, "Antofagasta (ATI)": 0.15,
        "Iquique": 0.15, "San Antonio": 0.10, "Ventanas": 0.10,
        "Barquito": 0.05, "Coquimbo": 0.05,
    },
    "blister": {
        "Mejillones": 0.40, "Ventanas": 0.30, "Barquito": 0.20, "Antofagasta (ATI)": 0.10,
    },
}

def classify_hs(code_str):
    """Map an HS code string to (commodity, product_form) or None."""
    code = str(code_str).replace(".", "").replace(" ", "").strip()
    # Remove trailing zeros that pad to 8 digits (e.g. 26030000 -> 2603)
    code_trimmed = code.rstrip("0") or code
    # Try exact prefixes at multiple lengths (longest first for specificity)
    for prefix_len in [8, 6, 4]:
        prefix = code[:prefix_len]
        if prefix in HS_MINERAL_MAP:
            return HS_MINERAL_MAP[prefix]
        # Also try trimmed version
        trimmed = code_trimmed[:prefix_len]
        if trimmed in HS_MINERAL_MAP:
            return HS_MINERAL_MAP[trimmed]
    return None

aduanas_loaded = False
aduanas_port_country = None   # will hold port x country x commodity FOB if loaded
PORT_PRODUCT_MAP = {}

if os.path.exists(SALIDAS_PATH):
    section_header(f"4A. ADUANAS PORT SHARES ({os.path.basename(SALIDAS_PATH)})")

    # ── Load with column discovery ────────────────────────────────────────
    # Aduanas CSVs typically use semicolons; sniff first line to detect
    with open(SALIDAS_PATH, "r", encoding="utf-8", errors="replace") as f:
        first_line = f.readline()
    _sep = ";" if first_line.count(";") > first_line.count(",") else ","
    print(f"  Detected delimiter: {'semicolon' if _sep == ';' else 'comma'}")

    sal = pd.read_csv(SALIDAS_PATH, sep=_sep, low_memory=False, dtype=str,
                       encoding="utf-8", on_bad_lines="skip")
    sal.columns = [c.strip().upper() for c in sal.columns]
    print(f"  Salidas loaded: {len(sal):,} rows x {len(sal.columns)} cols")
    print(f"  Source: {os.path.basename(SALIDAS_PATH)}")
    print(f"  Columns: {list(sal.columns)[:15]}...")

    # Filter to export operations only
    op_col = next((c for c in sal.columns if "TIPO_OPERACION" in c.upper()), None)
    if op_col:
        before = len(sal)
        sal = sal[sal[op_col].astype(str).str.strip().isin(EXPORT_OP_CODES)]
        print(f"  Export filter ({op_col}): {before:,} -> {len(sal):,} rows "
              f"(removed {before - len(sal):,} non-export operations)")
    else:
        print(f"  Warning: no operation type column found, skipping export filter")

    # Identify key columns by name patterns
    _col_map = {}
    for c in sal.columns:
        cl = c.lower()
        if "item_sa" in cl or "arancel" in cl:
            _col_map.setdefault("hs", c)
        if "puerto" in cl and "embarque" in cl:
            _col_map.setdefault("port", c)
        if "pais" in cl and ("destino" in cl or "origen" in cl):
            _col_map.setdefault("country", c)
        if "fob" in cl and "dolar" in cl:
            _col_map.setdefault("fob", c)
        if "fob" in cl and "fob" not in _col_map:
            _col_map.setdefault("fob", c)

    print(f"  Column mapping: {_col_map}")
    required = {"hs", "port", "country", "fob"}
    if required.issubset(_col_map.keys()):
        hs_col = _col_map["hs"]
        port_col = _col_map["port"]
        country_col = _col_map["country"]
        fob_col = _col_map["fob"]

        # ── Classify mineral rows ─────────────────────────────────────────
        sal["_hs_clean"] = sal[hs_col].astype(str).str.replace(".", "", regex=False).str.strip()
        sal["_mineral"] = sal["_hs_clean"].apply(classify_hs)
        mineral = sal[sal["_mineral"].notna()].copy()
        mineral["COMMODITY"] = mineral["_mineral"].apply(lambda x: x[0])
        mineral["PRODUCT_FORM"] = mineral["_mineral"].apply(lambda x: x[1])
        mineral["FOB_USD"] = pd.to_numeric(
            mineral[fob_col].str.replace(",", "."), errors="coerce").fillna(0)
        mineral["PORT_CODE"] = pd.to_numeric(mineral[port_col], errors="coerce")
        mineral["COUNTRY_CODE"] = pd.to_numeric(mineral[country_col], errors="coerce")

        print(f"  Mineral rows: {len(mineral):,} / {len(sal):,} ({len(mineral)/len(sal)*100:.1f}%)")
        print(f"  Total mineral FOB: ${mineral['FOB_USD'].sum():,.0f}")

        # Diagnostic: show potential lithium rows that didn't match
        li_candidates = sal[sal["_hs_clean"].str.startswith(("2836", "2825", "2842"), na=False)]
        li_unmatched = li_candidates[li_candidates["_mineral"].isna()]
        if len(li_unmatched) > 0:
            li_fob = pd.to_numeric(li_unmatched[fob_col], errors="coerce").sum()
            print(f"\n  Lithium HS diagnostic: {len(li_unmatched)} unmatched rows "
                  f"starting with 2836/2825/2842 (${li_fob:,.0f} FOB)")
            print(f"    Sample ITEM_SA values: {sorted(li_unmatched['_hs_clean'].unique()[:10])}")
        print(f"\n  By commodity:")
        for comm, grp in mineral.groupby("COMMODITY"):
            print(f"    {comm:<15} {len(grp):>6} rows  ${grp['FOB_USD'].sum():>15,.0f}")

        # ── Load all lookups from codigos ─────────────────────────────────
        country_lookup = {}
        port_lookup = {}     # code -> port name
        transport_lookup = {} # code -> transport mode
        region_lookup = {}   # code -> region name
        op_lookup = {}       # code -> operation name

        if os.path.exists(CODIGOS_PATH):
            country_lookup = parse_codigos_sheet(CODIGOS_PATH, "Países", "COD_PAIS", "NOMBRE_PAIS")
            port_lookup_raw = parse_codigos_sheet(CODIGOS_PATH, "Puertos", "COD_PUERTO", "NOMBRE_PUERTO")
            transport_lookup = parse_codigos_sheet(CODIGOS_PATH, "Vías de Transporte", "COD_VIA_TRANSPORTE", "NOMBRE_VIA_TRANSPORTE")
            region_lookup = parse_codigos_sheet(CODIGOS_PATH, "Regiones", "COD_REGION_ORIGEN", "NOMBRE_REGION")
            op_lookup = parse_codigos_sheet(CODIGOS_PATH, "Tipos de Operación", "COD_TIPO_OPERACION", "NOMBRE_TIPO_OPERACION")

            # Build port code -> pipeline name mapping from codigos
            # First use alias overrides, then title-case the codigos name
            for code, raw_name in port_lookup_raw.items():
                upper = raw_name.upper().strip()
                if upper in ADUANAS_PORT_ALIAS:
                    port_lookup[code] = ADUANAS_PORT_ALIAS[upper]
                else:
                    port_lookup[code] = raw_name.title()

            # Add manual codes not in codigos
            for code, name in ADUANAS_MANUAL_PORTS.items():
                if code not in port_lookup:
                    port_lookup[code] = name

            print(f"  Codigos lookups loaded:")
            print(f"    Countries: {len(country_lookup)}, Ports: {len(port_lookup)}, "
                  f"Transport modes: {len(transport_lookup)}, Regions: {len(region_lookup)}")
        else:
            print(f"  Warning: codigos file not found at {CODIGOS_PATH}")

        # ── Map port codes to names ───────────────────────────────────────
        mineral["PORT_NAME"] = mineral["PORT_CODE"].map(port_lookup)
        unmapped_ports = mineral[mineral["PORT_NAME"].isna()]["PORT_CODE"].dropna().unique()
        if len(unmapped_ports) > 0:
            for pc in sorted(unmapped_ports):
                vol = mineral[mineral["PORT_CODE"] == pc]["FOB_USD"].sum()
                if vol > 1_000_000:
                    print(f"  Warning: unmapped port code {int(pc)}, FOB ${vol:,.0f}")

        # ── Map transport mode ────────────────────────────────────────────
        via_col = next((c for c in sal.columns if "VIA_TRANSPORTE" in c.upper()), None)
        if via_col and transport_lookup:
            mineral["TRANSPORT_CODE"] = pd.to_numeric(mineral[via_col], errors="coerce")
            mineral["TRANSPORT_MODE"] = mineral["TRANSPORT_CODE"].map(transport_lookup)
            transport_summary = mineral.groupby("TRANSPORT_MODE")["FOB_USD"].sum().sort_values(ascending=False)
            print(f"\n  Transport modes (mineral FOB):")
            for mode, fob in transport_summary.items():
                print(f"    {mode:<30} ${fob:>15,.0f}  ({fob/mineral['FOB_USD'].sum()*100:.1f}%)")

        # ── Map origin region ─────────────────────────────────────────────
        reg_col = next((c for c in sal.columns if "REGION_ORIGEN" in c.upper()), None)
        if reg_col and region_lookup:
            mineral["REGION_CODE"] = pd.to_numeric(mineral[reg_col], errors="coerce")
            mineral["REGION_NAME"] = mineral["REGION_CODE"].map(region_lookup)

        # ── Map country codes to names ────────────────────────────────────
        mineral["COUNTRY_NAME"] = mineral["COUNTRY_CODE"].map(
            lambda c: country_lookup.get(int(c), f"Code_{int(c)}") if pd.notna(c) else None
        )
        # Normalize to English names used in COUNTRY_COORDS
        ADUANAS_COUNTRY_ALIAS = {
            **COUNTRY_ALIAS,
            "China": "China", "Japón": "Japan", "Alemania": "Germany",
            "España": "Spain", "Francia": "France", "Italia": "Italy",
            "Países Bajos": "Netherlands", "Bélgica": "Belgium",
            "Suecia": "Sweden", "Bulgaria": "Bulgaria", "Finlandia": "Finland",
            "Canadá": "Canada", "México": "Mexico", "Perú": "Peru",
            "Argentina": "Argentina", "Brasil": "Brazil",
            "Reino Unido": "United Kingdom", "Suiza": "Switzerland",
            "Grecia": "Greece", "Portugal": "Portugal",
            "India": "India", "Indonesia": "Indonesia",
            "Hong Kong": "Hong Kong", "Polonia": "Poland",
        }
        mineral["COUNTRY_EN"] = mineral["COUNTRY_NAME"].map(
            lambda x: ADUANAS_COUNTRY_ALIAS.get(x, x) if pd.notna(x) else None
        )

        # ── Compute PORT_PRODUCT_MAP from actual data ─────────────────────
        # For copper product forms, compute FOB-weighted port shares
        cu_mineral = mineral[mineral["COMMODITY"] == "Copper"]
        for product in ["concentrate", "cathode", "blister"]:
            prod_data = cu_mineral[cu_mineral["PRODUCT_FORM"] == product]
            prod_data = prod_data[prod_data["PORT_NAME"].notna()]
            total_fob = prod_data["FOB_USD"].sum()
            if total_fob > 0:
                shares = prod_data.groupby("PORT_NAME")["FOB_USD"].sum() / total_fob
                shares = shares[shares >= 0.005].to_dict()
                PORT_PRODUCT_MAP[product] = shares

        # For non-copper commodities, compute per-commodity port shares
        for comm in mineral["COMMODITY"].unique():
            if comm == "Copper":
                continue
            comm_data = mineral[(mineral["COMMODITY"] == comm) & (mineral["PORT_NAME"].notna())]
            total_fob = comm_data["FOB_USD"].sum()
            if total_fob > 0:
                shares = comm_data.groupby("PORT_NAME")["FOB_USD"].sum() / total_fob
                product_form = comm_data["PRODUCT_FORM"].mode().iloc[0] if len(comm_data) > 0 else "unknown"
                key = f"{comm.lower()}_{product_form}"
                PORT_PRODUCT_MAP[key] = shares[shares >= 0.005].to_dict()

        print(f"\n  PORT_PRODUCT_MAP derived for: {list(PORT_PRODUCT_MAP.keys())}")
        for key, shares in PORT_PRODUCT_MAP.items():
            top3 = sorted(shares.items(), key=lambda x: -x[1])[:3]
            top3_str = ", ".join(f"{p} {s*100:.1f}%" for p, s in top3)
            print(f"    {key:<30} {len(shares)} ports  (top: {top3_str})")

        # ── Build port x country x commodity aggregation for direct edges ─
        aduanas_port_country = mineral[
            mineral["PORT_NAME"].notna() & mineral["COUNTRY_EN"].notna()
        ].groupby(
            ["COMMODITY", "PRODUCT_FORM", "PORT_NAME", "COUNTRY_EN"]
        )["FOB_USD"].sum().reset_index()
        aduanas_port_country = aduanas_port_country[aduanas_port_country["FOB_USD"] > 0]
        print(f"\n  Aduanas port-country flows: {len(aduanas_port_country)} "
              f"({aduanas_port_country['COMMODITY'].nunique()} commodities, "
              f"{aduanas_port_country['PORT_NAME'].nunique()} ports, "
              f"{aduanas_port_country['COUNTRY_EN'].nunique()} countries)")

        # ── Save port shares CSV (for Section 7 comparison) ──────────────
        shares_rows = []
        for product in ["concentrate", "cathode", "blister"]:
            if product in PORT_PRODUCT_MAP:
                for port, share in PORT_PRODUCT_MAP[product].items():
                    shares_rows.append({"PRODUCT": product, "PORT": port, "FOB_SHARE": share})
        if shares_rows:
            pd.DataFrame(shares_rows).to_csv(
                os.path.join(DIR_PRELIM, "Chile_Port_Shares_Aduanas.csv"), index=False)
            print(f"  Saved: Chile_Port_Shares_Aduanas.csv ({len(shares_rows)} rows)")

        aduanas_loaded = True
        mineral.drop(columns=["_hs_clean", "_mineral"], inplace=True)
    else:
        print(f"  ERROR: Could not map required columns. Found: {_col_map}")
        print(f"  Available columns: {list(sal.columns)}")

else:
    print(f"\n  Salidas CSV not found at {SALIDAS_PATH}")
    print("  Using fallback hardcoded PORT_PRODUCT_MAP")

# Apply fallback for any missing product types
for key, fallback in PORT_PRODUCT_MAP_FALLBACK.items():
    if key not in PORT_PRODUCT_MAP:
        PORT_PRODUCT_MAP[key] = fallback
        if aduanas_loaded:
            print(f"  Fallback applied for: {key}")

# ── Build port-to-country edges ──────────────────────────────────────────

def build_export_edges(dest_data, commodity, product_form, unit, port_shares):
    """Build edges from COCHILCO destination data distributed across ports."""
    result_edges = []
    for country_raw, info in dest_data.items():
        country = normalize_country(country_raw)
        coords = COUNTRY_COORDS.get(country)
        if not coords:
            continue
        value = info["value"]
        for port_name, share in port_shares.items():
            port = next((p for p in ports_df.to_dict("records") if p["name"] == port_name), None)
            if not port:
                continue
            result_edges.append({
                "FROM_NAME": port_name, "FROM_TYPE": "port",
                "FROM_LAT": port["lat"], "FROM_LON": port["lon"],
                "TO_NAME": country, "TO_TYPE": "country",
                "TO_LAT": coords["lat"], "TO_LON": coords["lon"],
                "EDGE_TYPE": "port_to_country", "PRODUCT_FORM": product_form,
                "COMMODITIES": commodity, "DISTANCE_KM": None,
                "EXPORT_VALUE": round(value * share, 2),
                "EXPORT_UNIT": unit, "DESTINATION_TOTAL": round(value, 2),
            })
    return result_edges

def build_aduanas_edges(apc_df, commodity, product_form):
    """Build edges directly from Aduanas port x country aggregation (FOB USD)."""
    result_edges = []
    sub = apc_df[(apc_df["COMMODITY"] == commodity) & (apc_df["PRODUCT_FORM"] == product_form)]
    port_dict = {p["name"]: p for p in ports_df.to_dict("records")}
    for _, row in sub.iterrows():
        country = row["COUNTRY_EN"]
        coords = COUNTRY_COORDS.get(country)
        port = port_dict.get(row["PORT_NAME"])
        if not coords or not port:
            continue
        result_edges.append({
            "FROM_NAME": row["PORT_NAME"], "FROM_TYPE": "port",
            "FROM_LAT": port["lat"], "FROM_LON": port["lon"],
            "TO_NAME": country, "TO_TYPE": "country",
            "TO_LAT": coords["lat"], "TO_LON": coords["lon"],
            "EDGE_TYPE": "port_to_country", "PRODUCT_FORM": product_form,
            "COMMODITIES": commodity, "DISTANCE_KM": None,
            "EXPORT_VALUE": round(row["FOB_USD"], 2),
            "EXPORT_UNIT": "$FOB",
            "DESTINATION_TOTAL": round(row["FOB_USD"], 2),
        })
    return result_edges

# Copper and COCHILCO-covered commodities: use COCHILCO volumes x Aduanas port shares
export_edges = []
export_edges.extend(build_export_edges(
    cu_concentrate, "Copper", "concentrate", "kMT", PORT_PRODUCT_MAP["concentrate"]))
export_edges.extend(build_export_edges(
    cu_refined, "Copper", "cathode", "kMT", PORT_PRODUCT_MAP["cathode"]))
export_edges.extend(build_export_edges(
    cu_blister, "Copper", "blister", "kMT", PORT_PRODUCT_MAP["blister"]))
export_edges.extend(build_export_edges(
    mo_concentrate, "Molybdenum", "concentrate", "MT",
    PORT_PRODUCT_MAP.get("molybdenum_mo_concentrate",
                         {"Mejillones": 0.50, "Antofagasta (ATI)": 0.30, "Barquito": 0.20})))
export_edges.extend(build_export_edges(
    li_exports, "Lithium", "lithium_compounds", "$M_FOB",
    PORT_PRODUCT_MAP.get("lithium_lithium_compounds",
                         {"Antofagasta (ATI)": 0.50, "Mejillones": 0.30, "Iquique": 0.20})))
export_edges.extend(build_export_edges(
    io_exports, "Iodine", "iodine", "$M_FOB",
    PORT_PRODUCT_MAP.get("iodine_iodine",
                         {"Iquique": 0.40, "Patache": 0.30, "Antofagasta (ATI)": 0.20, "Mejillones": 0.10})))

# Non-COCHILCO commodities: build edges directly from Aduanas if available
ADUANAS_ONLY_COMMODITIES = [
    ("Gold", "gold_refined"), ("Silver", "silver_refined"),
    ("Iron", "iron_ore"), ("Manganese", "mn_ore"),
    ("Boron", "borate"), ("Nitrate", "nitrate"), ("Rhenium", "perrhenate"),
]
if aduanas_loaded and aduanas_port_country is not None:
    for comm, pform in ADUANAS_ONLY_COMMODITIES:
        new_edges = build_aduanas_edges(aduanas_port_country, comm, pform)
        if new_edges:
            export_edges.extend(new_edges)
            total_fob = sum(e["EXPORT_VALUE"] for e in new_edges)
            n_countries = len(set(e["TO_NAME"] for e in new_edges))
            n_ports = len(set(e["FROM_NAME"] for e in new_edges))
            print(f"  {comm:<15} {len(new_edges):>4} edges  "
                  f"({n_countries} countries, {n_ports} ports, ${total_fob:,.0f} FOB)")

export_df = pd.DataFrame(export_edges)

print(f"\nExport edges: {len(export_df)}")
for commodity in export_df["COMMODITIES"].unique():
    sub = export_df[export_df["COMMODITIES"] == commodity]
    print(f"  {commodity:<15} {len(sub):>5} edges ({sub['TO_NAME'].nunique()} countries, {sub['FROM_NAME'].nunique()} ports)")

# Append to unified edge table
edges = edges[edges["EDGE_TYPE"] != "port_to_country"]  # remove old if re-run
export_aligned = export_df[common_cols].copy()
for col in common_cols:
    if col not in export_aligned.columns:
        export_aligned[col] = ""
edges = pd.concat([edges, export_aligned], ignore_index=True)

print(f"\nUnified edges: {len(edges)}")
for et, count in edges["EDGE_TYPE"].value_counts().items():
    print(f"  {et:<25} {count:>5}")


# %% 5. CLEANUP & VALIDATION

# ── 5A. Smelter name standardization ─────────────────────────────────────

section_header("5A. SMELTER NAME STANDARDIZATION")

print("Name mapping verification:")
for canonical, inv_name in SMELTER_NAME_MAP.items():
    exists = inv["FACILITY_NAME"].eq(inv_name).any()
    print(f"  {canonical:<35} -> {inv_name:<55} {'OK' if exists else 'NOT FOUND'}")

renamed_count = 0
for canonical, inv_name in SMELTER_NAME_MAP.items():
    if canonical == inv_name:
        continue
    for col in ["FROM_NAME", "TO_NAME"]:
        mask = edges[col] == canonical
        if mask.any():
            edges.loc[mask, col] = inv_name
            renamed_count += mask.sum()
            print(f"  Renamed {col}: '{canonical}' -> '{inv_name}' ({mask.sum()} edges)")

print(f"\nTotal renames: {renamed_count}")

# Handle Las Ventanas / Ventanas ambiguity
las_ventanas = inv[inv["FACILITY_NAME"].str.contains("Las Ventanas", case=False, na=False)]
ventanas_plain = inv[inv["FACILITY_NAME"].str.contains("Ventanas refinery", case=False, na=False) &
                     ~inv["FACILITY_NAME"].str.contains("Las Ventanas", case=False, na=False)]

if len(las_ventanas) > 0 and len(ventanas_plain) > 0:
    print(f"\n  Note: Both '{las_ventanas.iloc[0]['FACILITY_NAME']}' AND "
          f"'{ventanas_plain.iloc[0]['FACILITY_NAME']}' exist. Leaving both.")
elif len(las_ventanas) > 0:
    lv_name = las_ventanas.iloc[0]["FACILITY_NAME"]
    for col in ["FROM_NAME", "TO_NAME"]:
        mask = edges[col].str.contains("Ventanas refinery", case=False, na=False) & \
               ~edges[col].str.contains("Las Ventanas", case=False, na=False)
        if mask.any():
            edges.loc[mask, col] = lv_name

# ── 5B. Andacollo Oro mine link ──────────────────────────────────────────

section_header("5B. ANDACOLLO ORO MINE")

andacollo = inv[inv["FACILITY_NAME"].str.contains("Andacollo", case=False, na=False)]
print("  Andacollo-related facilities:")
for _, row in andacollo.iterrows():
    prod = row.get("COCHILCO_CU_2024_KMT", np.nan)
    prod_str = f"{prod:.1f} kMT" if pd.notna(prod) else ""
    print(f"    {row['FACILITY_NAME']:<45} {row['FACILITY_TYPE']:<20} {prod_str}")

existing_anda = links[links["MINE_NAME"].str.contains("Andacollo", case=False, na=False)]
if len(existing_anda) > 0:
    print(f"  Link already exists: {existing_anda.iloc[0]['MINE_NAME']} -> {existing_anda.iloc[0]['PLANT_NAME']}")
else:
    anda_mine = inv[inv["FACILITY_NAME"].str.contains("Andacollo", case=False, na=False) &
                    inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]
    if len(anda_mine) > 0:
        mine_row = anda_mine.iloc[0]
        mine_lat, mine_lon = mine_row["LATITUD"], mine_row["LONGITUD"]
        if pd.notna(mine_lat):
            sxew_plants = inv[(inv["CHAIN_STAGE"] == "sx_ew") & inv["LATITUD"].notna()].copy()
            sxew_plants["_dist"] = sxew_plants.apply(
                lambda r: haversine_km(mine_lat, mine_lon, r["LATITUD"], r["LONGITUD"]), axis=1)
            nearby = sxew_plants[sxew_plants["_dist"] < 50].sort_values("_dist")
            if len(nearby) > 0:
                target = nearby.iloc[0]
                new_link = {
                    "MINE_NAME": mine_row["FACILITY_NAME"],
                    "PLANT_NAME": target["FACILITY_NAME"],
                    "MINE_LAT": mine_lat, "MINE_LON": mine_lon,
                    "PLANT_LAT": target["LATITUD"], "PLANT_LON": target["LONGITUD"],
                    "SHARED_COMMODITIES": "Copper",
                    "DISTANCE_KM": round(target["_dist"], 1),
                    "PRODUCT_FORM": "cathode_sxew",
                }
                links = pd.concat([links, pd.DataFrame([new_link])], ignore_index=True)
                new_edge = {
                    "FROM_NAME": mine_row["FACILITY_NAME"], "FROM_TYPE": "mine",
                    "FROM_LAT": mine_lat, "FROM_LON": mine_lon,
                    "TO_NAME": target["FACILITY_NAME"], "TO_TYPE": "plant",
                    "TO_LAT": target["LATITUD"], "TO_LON": target["LONGITUD"],
                    "EDGE_TYPE": "mine_to_plant", "PRODUCT_FORM": "cathode_sxew",
                    "COMMODITIES": "Copper", "DISTANCE_KM": round(target["_dist"], 1),
                }
                edges = pd.concat([edges, pd.DataFrame([new_edge])], ignore_index=True)
                print(f"  -> Created link + edge: {mine_row['FACILITY_NAME']} -> {target['FACILITY_NAME']}")
            else:
                print("  No SX-EW plant within 50 km.")

# ── 5C. Deduplication ────────────────────────────────────────────────────

section_header("5C. DEDUPLICATION")

before = len(edges)
edges = edges.drop_duplicates(subset=["FROM_NAME", "TO_NAME", "EDGE_TYPE", "COMMODITIES", "PRODUCT_FORM"], keep="first")
print(f"  Removed {before - len(edges)} duplicate edges ({before} -> {len(edges)})")

# ── 5D. Validation ───────────────────────────────────────────────────────

section_header("5D. VALIDATION")

issues = []

# Production coverage
cu_matched_total = inv["COCHILCO_CU_2024_KMT"].sum()
cu_count = inv["COCHILCO_CU_2024_KMT"].notna().sum()
print(f"  Cu production: {cu_count} records, {cu_matched_total:,.1f} / {cu_total:,.1f} kMT "
      f"({cu_matched_total/cu_total*100:.1f}%)")

# Path traceability
print(f"\n  Path traceability:")
cu_producers = inv[(inv["COCHILCO_CU_2024_KMT"].notna()) & (inv["COCHILCO_CU_2024_KMT"] > 0)]
connected_prod = 0
disconnected = []

for _, mrow in cu_producers.iterrows():
    mine_name = mrow["FACILITY_NAME"]
    mine_edges = edges[(edges["EDGE_TYPE"] == "mine_to_plant") & (edges["FROM_NAME"] == mine_name)]
    if len(mine_edges) == 0:
        mine_edges = edges[(edges["EDGE_TYPE"] == "mine_to_plant") &
                           edges["FROM_NAME"].str.contains(mine_name[:8], case=False, na=False, regex=False)]

    if len(mine_edges) == 0:
        disconnected.append((mine_name, mrow["COCHILCO_CU_2024_KMT"]))
        continue

    plants = set(mine_edges["TO_NAME"])
    reaches_port = False
    for plant in plants:
        if edges[(edges["FROM_NAME"] == plant) &
                 (edges["EDGE_TYPE"].isin(["concentrate_to_port", "sxew_to_port", "smelter_to_port"]))].shape[0] > 0:
            reaches_port = True
            break
        smelter_edges = edges[(edges["FROM_NAME"] == plant) & (edges["EDGE_TYPE"] == "concentrate_to_smelter")]
        for _, se in smelter_edges.iterrows():
            if edges[(edges["FROM_NAME"] == se["TO_NAME"]) & (edges["EDGE_TYPE"] == "smelter_to_port")].shape[0] > 0:
                reaches_port = True
                break
        if reaches_port:
            break

    if reaches_port:
        connected_prod += mrow["COCHILCO_CU_2024_KMT"]
    else:
        disconnected.append((mine_name, mrow["COCHILCO_CU_2024_KMT"]))

print(f"    Mines reaching port: {cu_count - len(disconnected)} / {cu_count}")
print(f"    Production reaching ports: {connected_prod:,.1f} / {cu_matched_total:,.1f} kMT "
      f"({connected_prod/cu_matched_total*100:.1f}%)")

if disconnected:
    print(f"    Disconnected mines:")
    for name, prod in sorted(disconnected, key=lambda x: -x[1]):
        print(f"      {name:<45} {prod:>8.1f} kMT")
    issues.append(f"{len(disconnected)} mines ({sum(p for _,p in disconnected):,.1f} kMT) don't reach a port")

# Port summary
print(f"\n  Port summary:")
for etype in ["concentrate_to_port", "smelter_to_port", "sxew_to_port"]:
    sub = edges[edges["EDGE_TYPE"] == etype]
    ports_out = sub["TO_NAME"].unique()
    print(f"    {etype:<25} -> {', '.join(sorted(ports_out))}")

port_to_country = edges[edges["EDGE_TYPE"] == "port_to_country"]
print(f"    port_to_country: {port_to_country['FROM_NAME'].nunique()} ports -> "
      f"{port_to_country['TO_NAME'].nunique()} countries ({len(port_to_country)} edges)")

# ── 5E. Save all files ───────────────────────────────────────────────────

section_header("5E. SAVE")

edges.sort_values(["EDGE_TYPE", "FROM_NAME", "TO_NAME"], inplace=True)
edges.reset_index(drop=True, inplace=True)

for et, count in edges["EDGE_TYPE"].value_counts().sort_index().items():
    print(f"  {et:<25} {count:>5}")
print(f"  {'TOTAL':<25} {len(edges):>5}")

if issues:
    print(f"\nRemaining issues ({len(issues)}):")
    for i, iss in enumerate(issues, 1):
        print(f"  {i}. {iss}")

# Save
ds = edges[edges["EDGE_TYPE"] != "mine_to_plant"]
export_df.to_csv(os.path.join(DIR_PRELIM, "Chile_Export_Destinations.csv"), index=False)
edges.to_csv(os.path.join(DIR_PRELIM, "Chile_Supply_Chain_Edges.csv"), index=False)
inv.to_csv(inv_path, index=False)
links.to_csv(links_path, index=False)
ds.to_csv(os.path.join(DIR_PRELIM, "Chile_Downstream_Links.csv"), index=False)

print(f"\nSaved:")
print(f"  Chile_Minerals_Inventory.csv   ({len(inv)} records)")
print(f"  Chile_Mine_Plant_Links.csv     ({len(links)} links)")
print(f"  Chile_Supply_Chain_Edges.csv   ({len(edges)} edges)")
print(f"  Chile_Downstream_Links.csv     ({len(ds)} downstream edges)")
print(f"  Chile_Export_Destinations.csv   ({len(export_df)} edges)")
print(f"  Chile_Ports.csv                ({len(ports_df)} ports)")


# %% 6. DIAGNOSTICS (merged)

section_header("DIAGNOSTIC: CHECK 1 - MULTI-MATCH AUDIT")

multi_match_cases = []
plant_attribution_cases = []

for company, search_terms in COMPANY_TO_DEPOSIT.items():
    for term in search_terms:
        all_matches = inv[inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False)]
        mine_matches = all_matches[all_matches["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]
        other_matches = all_matches[~all_matches["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]

        if len(all_matches) > 1:
            multi_match_cases.append({"company": company, "term": term,
                "total_matches": len(all_matches), "mines": len(mine_matches), "non_mines": len(other_matches)})
            print(f"  {company} ('{term}'): {len(all_matches)} matches "
                  f"({len(mine_matches)} mines, {len(other_matches)} non-mines)")
            for _, row in all_matches.iterrows():
                selected = " <-- SELECTED" if (len(mine_matches) > 0 and row.name == mine_matches.index[0]) else ""
                if len(mine_matches) == 0 and row.name == all_matches.index[0]:
                    selected = " <-- SELECTED"
                print(f"    {row['FACILITY_NAME']:<50} {row['FACILITY_TYPE']:<20}{selected}")

        if len(mine_matches) == 0 and len(all_matches) > 0:
            target = all_matches.iloc[0]
            plant_attribution_cases.append({"company": company,
                "facility": target["FACILITY_NAME"], "facility_type": target["FACILITY_TYPE"]})

        if len(all_matches) > 0:
            break

print(f"\nSummary: {len(multi_match_cases)} companies have multiple matches")

# ── CHECK 2: Production on non-mine records ──────────────────────────────

section_header("DIAGNOSTIC: CHECK 2 - NON-MINE PRODUCTION")

cu_prod = inv[inv["COCHILCO_CU_2024_KMT"].notna()].copy()
non_mine_prod = cu_prod[~cu_prod["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]
print(f"  Total records with Cu production: {len(cu_prod)}")
print(f"  Records that ARE mines: {len(cu_prod) - len(non_mine_prod)}")
print(f"  Records that are NOT mines: {len(non_mine_prod)}")

if len(non_mine_prod) > 0:
    total_non_mine_kmt = 0
    for _, row in non_mine_prod.iterrows():
        prod = row["COCHILCO_CU_2024_KMT"]
        total_non_mine_kmt += prod
        print(f"  {row['FACILITY_NAME']:<45} {row['FACILITY_TYPE']:<20} {prod:>8.1f} kMT")
    print(f"\n  Total non-mine production: {total_non_mine_kmt:,.1f} kMT "
          f"({total_non_mine_kmt/cu_prod['COCHILCO_CU_2024_KMT'].sum()*100:.1f}%)")

# ── CHECK 3: Edge matching method breakdown ──────────────────────────────

section_header("DIAGNOSTIC: CHECK 3 - EDGE MATCHING METHODS")

for etype in edges["EDGE_TYPE"].unique():
    sub = edges[edges["EDGE_TYPE"] == etype]
    if etype == "mine_to_plant":
        method = "distance (haversine + shared commodity)"
    elif etype == "concentrate_to_smelter":
        method = "mixed (link tracing + regional proximity)"
    elif etype == "concentrate_to_port":
        method = "distance (nearest port), except Escondida->Coloso"
    elif etype == "smelter_to_port":
        method = "direct (hardcoded smelter->port mapping)"
    elif etype == "sxew_to_port":
        method = "distance (nearest cathode-handling port)"
    elif etype == "port_to_country":
        method = "direct (COCHILCO export tables + port share weights)"
    else:
        method = "unknown"
    print(f"  {etype:<25} {len(sub):>5} edges  [{method}]")

# ── CHECK 4: Non-copper downstream coverage ──────────────────────────────

section_header("DIAGNOSTIC: CHECK 4 - NON-COPPER DOWNSTREAM")

downstream_check = edges[edges["EDGE_TYPE"] != "mine_to_plant"]
if "COMMODITIES" in downstream_check.columns:
    for comm, count in downstream_check["COMMODITIES"].value_counts().items():
        print(f"  {comm:<20} {count:>5} downstream edges")

print(f"\nMine-to-plant links by commodity:")
if "SHARED_COMMODITIES" in links.columns:
    comm_counts = Counter()
    for comms in links["SHARED_COMMODITIES"].dropna():
        for c in str(comms).split(","):
            c = c.strip()
            if c:
                comm_counts[c] += 1
    for c, count in comm_counts.most_common():
        # Check actual downstream edge presence for this commodity
        ds_edges = downstream_check[downstream_check["COMMODITIES"].str.contains(c, case=False, na=False)]
        p2c_edges = ds_edges[ds_edges["EDGE_TYPE"] == "port_to_country"]
        domestic_ds = ds_edges[ds_edges["EDGE_TYPE"] != "port_to_country"]
        if len(domestic_ds) > 0 and len(p2c_edges) > 0:
            has_ds = "YES (full chain)"
        elif len(p2c_edges) > 0:
            has_ds = "port_to_country only"
        elif len(domestic_ds) > 0:
            has_ds = "domestic only"
        else:
            has_ds = "NO"
        print(f"  {c:<20} {count:>5} links  downstream: {has_ds}")

# ── CHECK 5: Operator matching quality ────────────────────────────────────

section_header("DIAGNOSTIC: CHECK 5 - OPERATOR MATCHING")

def classify_operator_match(row):
    mine_op = str(row.get("MINE_OPERATOR", "")).strip()
    plant_op = str(row.get("PLANT_OPERATOR", "")).strip()
    if not mine_op or mine_op == "nan":
        return "no_mine_operator"
    if not plant_op or plant_op == "nan":
        return "no_plant_operator"
    if mine_op.lower() == plant_op.lower():
        return "same_operator"
    if mine_op.lower()[:8] in plant_op.lower() or plant_op.lower()[:8] in mine_op.lower():
        return "partial_match"
    return "different_operator"

if "MINE_OPERATOR" in links.columns and "PLANT_OPERATOR" in links.columns:
    links["_op_match"] = links.apply(classify_operator_match, axis=1)
    print("Link classification:")
    for cat, sub in links.groupby("_op_match"):
        median_dist = sub["DISTANCE_KM"].median() if "DISTANCE_KM" in sub.columns else 0
        print(f"  {cat:<25} {len(sub):>5} ({len(sub)/len(links)*100:>5.1f}%)  median dist: {median_dist:.0f} km")

    cross = links[links["_op_match"] == "different_operator"]
    if len(cross) > 0:
        print(f"\n  Cross-company links by distance:")
        for label, lo, hi in [("<20 km", 0, 20), ("20-50 km", 20, 50),
                               ("50-100 km", 50, 100), ("100-150 km", 100, 150), ("150+ km", 150, 9999)]:
            n = len(cross[(cross["DISTANCE_KM"] >= lo) & (cross["DISTANCE_KM"] < hi)])
            print(f"    {label:<18} {n:>5}")

    links.drop(columns=["_op_match"], inplace=True, errors="ignore")
else:
    print("  MINE_OPERATOR / PLANT_OPERATOR columns not found in links")

# ── CHECK 6: Smelter connectivity ─────────────────────────────────────────

section_header("DIAGNOSTIC: CHECK 6 - SMELTER CONNECTIVITY")

smelting_facilities = inv[inv["CHAIN_STAGE"] == "smelting"]["FACILITY_NAME"].unique()
print(f"Smelting facilities in inventory ({len(smelting_facilities)}):")

for name in sorted(smelting_facilities):
    as_m2p_dest = len(edges[(edges["EDGE_TYPE"] == "mine_to_plant") & (edges["TO_NAME"] == name)])
    as_c2s_dest = len(edges[(edges["EDGE_TYPE"] == "concentrate_to_smelter") & (edges["TO_NAME"] == name)])
    as_s2p_src  = len(edges[(edges["EDGE_TYPE"] == "smelter_to_port") & (edges["FROM_NAME"] == name)])

    connections = []
    if as_m2p_dest > 0: connections.append(f"m2p_dest={as_m2p_dest}")
    if as_c2s_dest > 0: connections.append(f"c2s_dest={as_c2s_dest}")
    if as_s2p_src > 0:  connections.append(f"s2p_src={as_s2p_src}")
    if not connections:  connections.append("DISCONNECTED")
    print(f"  {name:<60} [{', '.join(connections)}]")

print(f"\n{'=' * 65}")
print("DIAGNOSTICS COMPLETE")
print("=" * 65)
print(f"\nNote: {len(idle_mines)} idle mines retained in inventory but excluded from "
      f"all supply chain edges ({n_idle_links} phantom links removed).")


# %% 7. PORT DISTANCE COMPARISON

section_header("7. DISTANCE-BASED PORT ASSIGNMENT ANALYSIS")

# Extract mines with production and coordinates
mines = inv[
    (inv["CHAIN_STAGE"] == "extraction") &
    (inv["LATITUD"].notna()) &
    (inv["LONGITUD"].notna()) &
    (inv["COCHILCO_CU_2024_KMT"].notna()) &
    (inv["COCHILCO_CU_2024_KMT"] > 0)
].copy()

print(f"Mines with production data: {len(mines)}")
print(f"Ports: {len(ports_df)}")

# ── Product type: identify each mine's OWN processing facilities by name ───
#
# The links table connects mines to ALL nearby plants by distance, so link
# counts reflect geographic density of plants, not a mine's actual output.
# Instead, we look for plants in the inventory whose name contains the mine
# name (e.g. "Escondida concentrator" for Escondida mine) and classify from
# the CHAIN_STAGE of those named facilities.
#
# Priority: concentration > smelting > sx_ew > processing > fallback to links

# Manual overrides for mines where the inventory is outdated or incomplete.
# Spence: SGOP concentrator (2021+), primary output is now concentrate.
# Quebrada Blanca: QB2 concentrator (2023+), primary output is concentrate.
# Centinela: sulfide concentrator operational, concentrate is primary.
# Sierra Gorda: KGHM concentrator, concentrate.
# Caserones: Lumina concentrator, concentrate.
# Andina: Codelco concentrator (no named plant in inventory), concentrate.
# Ministro Hales: Codelco concentrator, concentrate.
# Las Luces (Las Cenizas): small oxide/SX-EW, cathode.
PRODUCT_TYPE_OVERRIDE = {
    "Spence": "concentrate",
    "Quebrada Blanca": "concentrate",
    "Centinela": "concentrate",
    "Sierra Gorda": "concentrate",
    "Caserones": "concentrate",
    "Andina": "concentrate",
    "Ministro Hales": "concentrate",
    "Las Luces": "cathode",
}

def classify_mine_product(mine_name, inv_df, links_df):
    """Determine product type from named processing facilities in inventory."""
    # Check manual overrides first
    for key, ptype in PRODUCT_TYPE_OVERRIDE.items():
        if key.lower() in mine_name.lower():
            return ptype

    # Build search terms from the mine name
    tokens = mine_name.split()
    search_terms = []
    if len(tokens) >= 2 and tokens[0] in ("El", "Los", "Las", "La"):
        search_terms.append(" ".join(tokens[:2]))
    search_terms.append(tokens[0] if len(tokens[0]) >= 5 else mine_name)

    named_plants = pd.DataFrame()
    for term in search_terms:
        candidates = inv_df[
            inv_df["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False) &
            ~inv_df["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False)
        ]
        if len(candidates) > 0:
            named_plants = candidates
            break

    if len(named_plants) > 0:
        stages = set(named_plants["CHAIN_STAGE"].dropna())
        if "concentration" in stages:
            return "concentrate"
        if "smelting" in stages:
            return "concentrate"
        if "processing" in stages:
            return "concentrate"
        if "sx_ew" in stages:
            return "cathode"

    # Fallback: use the mode of links
    if "MINE_NAME" in links_df.columns and "PRODUCT_FORM" in links_df.columns:
        mine_links = links_df[links_df["MINE_NAME"] == mine_name]
        if len(mine_links) > 0:
            mode = mine_links["PRODUCT_FORM"].value_counts().index[0]
            ptype_map = {"concentrate": "concentrate", "cathode_sxew": "cathode",
                         "cathode_er": "cathode", "blister": "blister"}
            return ptype_map.get(mode, "unknown")

    return "unknown"

mines["product_type"] = mines["FACILITY_NAME"].apply(
    lambda name: classify_mine_product(name, inv, links)
)

print(f"\nMines by product type:")
print(mines["product_type"].value_counts().to_string())

# Distance matrix
distances = np.zeros((len(mines), len(ports_df)))
for i, (_, mine) in enumerate(mines.iterrows()):
    for j, (_, port) in enumerate(ports_df.iterrows()):
        distances[i, j] = haversine_km(mine["LATITUD"], mine["LONGITUD"], port["lat"], port["lon"])

distance_df = pd.DataFrame(distances, index=mines["FACILITY_NAME"].values,
                           columns=ports_df["name"].values)
print(f"\nDistance matrix: {distance_df.shape}")
print(f"Distance range: {distances.min():.0f} - {distances.max():.0f} km")

# Assign nearest port
mines["nearest_port"] = distance_df.idxmin(axis=1).values
mines["distance_km"] = distance_df.min(axis=1).values

print(f"\nTop 10 mines with optimal port:")
for _, mine in mines.nlargest(10, "COCHILCO_CU_2024_KMT").iterrows():
    print(f"  {mine['FACILITY_NAME']:<40} -> {mine['nearest_port']:<25} "
          f"({mine['distance_km']:.0f} km, {mine['COCHILCO_CU_2024_KMT']:.1f} kMT)")

# Simulated shares
simulated_shares = {}
for product_type in ["concentrate", "cathode", "blister"]:
    product_mines = mines[mines["product_type"] == product_type]
    if len(product_mines) == 0:
        continue
    total_production = product_mines["COCHILCO_CU_2024_KMT"].sum()
    port_production = product_mines.groupby("nearest_port")["COCHILCO_CU_2024_KMT"].sum()
    port_shares = (port_production / total_production).to_dict()
    port_shares = {k: v for k, v in port_shares.items() if v >= 0.01}
    simulated_shares[product_type] = port_shares
    print(f"\n{product_type.upper()} (simulated):")
    for port, share in sorted(port_shares.items(), key=lambda x: -x[1]):
        print(f"  {port:<30} {share*100:>6.1f}%")

# ── Load and harmonize actual port shares ─────────────────────────────────

port_shares_path = os.path.join(DIR_PRELIM, "Chile_Port_Shares_Aduanas.csv")
if os.path.exists(port_shares_path):
    actual_shares_df = pd.read_csv(port_shares_path)

    # FIX: Harmonize port names between Aduanas and pipeline
    ADUANAS_PORT_MAP = {
        "Caleta Coloso": "Coloso",
        "Puerto Angamos": "Angamos",
        "Antofagasta": "Antofagasta (ATI)",
        "Chañaral/Barquito": "Barquito",
    }
    actual_shares_df["PORT"] = actual_shares_df["PORT"].replace(ADUANAS_PORT_MAP)

    actual_shares = {}
    for product in ["concentrate", "cathode", "blister"]:
        product_data = actual_shares_df[actual_shares_df["PRODUCT"] == product]
        if len(product_data) == 0:
            continue
        port_share_dict = {}
        for _, row in product_data.iterrows():
            port_name = row["PORT"]
            # Aggregate after renaming (e.g., multiple rows mapping to same port)
            if port_name in port_share_dict:
                port_share_dict[port_name] += row["FOB_SHARE"]
            else:
                port_share_dict[port_name] = row["FOB_SHARE"]
        port_share_dict = {k: v for k, v in port_share_dict.items() if v >= 0.01}
        actual_shares[product] = port_share_dict

        print(f"\n{product.upper()} (actual, Aduanas):")
        for port, share in sorted(port_share_dict.items(), key=lambda x: -x[1]):
            print(f"  {port:<30} {share*100:>6.1f}%")

    # Comparison
    section_header("COMPARISON: ACTUAL vs DISTANCE-OPTIMAL")

    comparison_results = []
    for product_type in ["concentrate", "cathode", "blister"]:
        if product_type not in simulated_shares:
            continue

        actual = actual_shares.get(product_type, {})
        simulated = simulated_shares[product_type]
        all_ports = set(actual.keys()) | set(simulated.keys())

        print(f"\n{product_type.upper()}")
        print("-" * 75)

        for port in sorted(all_ports):
            actual_share = actual.get(port, 0)
            simulated_share = simulated.get(port, 0)
            diff = simulated_share - actual_share

            comparison_results.append({
                "product": product_type, "port": port,
                "actual_share": actual_share, "simulated_share": simulated_share,
                "difference": diff,
            })

            if abs(diff) < 0.02:
                indicator = "="
            elif diff > 0:
                indicator = "^ (would gain)"
            else:
                indicator = "v (would lose)"

            print(f"  {port:<30} Actual: {actual_share*100:5.1f}%  |  "
                  f"Optimal: {simulated_share*100:5.1f}%  |  D {diff*100:+6.1f}% {indicator}")

    comparison_df = pd.DataFrame(comparison_results)

    # Summary stats
    section_header("SUMMARY STATISTICS")
    for product_type in ["concentrate", "cathode", "blister"]:
        product_comp = comparison_df[comparison_df["product"] == product_type]
        if len(product_comp) == 0:
            continue
        mad = product_comp["difference"].abs().mean()
        total_mismatch = product_comp["difference"].abs().sum() / 2
        print(f"\n{product_type.upper()}:")
        print(f"  Mean Absolute Difference: {mad*100:.1f}%")
        print(f"  Total Volume Mismatch: {total_mismatch*100:.1f}%")

    # Save comparison outputs
    comparison_df.to_csv(os.path.join(DIR_PRELIM, "Port_Distance_Comparison.csv"), index=False)
    mines[["FACILITY_NAME", "COCHILCO_CU_2024_KMT", "product_type", "nearest_port", "distance_km"]].to_csv(
        os.path.join(DIR_PRELIM, "Mine_Optimal_Port_Assignments.csv"), index=False)
    distance_df.to_csv(os.path.join(DIR_PRELIM, "Mine_Port_Distance_Matrix.csv"))
    print(f"\nSaved: Port_Distance_Comparison.csv, Mine_Optimal_Port_Assignments.csv, Mine_Port_Distance_Matrix.csv")

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle("Port Share Comparison: Actual vs Distance-Optimal", fontsize=16, fontweight="bold")

    products = ["concentrate", "cathode", "blister"]
    colors = {"actual": "#FF6B6B", "simulated": "#4ECDC4"}

    for idx, product in enumerate(products):
        ax = axes[idx]
        if product not in simulated_shares:
            ax.text(0.5, 0.5, f"No {product} data", ha="center", va="center", transform=ax.transAxes)
            continue

        product_data = comparison_df[comparison_df["product"] == product].copy()
        product_data = product_data.sort_values("actual_share", ascending=False)
        product_data = product_data[
            (product_data["actual_share"] >= 0.01) | (product_data["simulated_share"] >= 0.01)
        ].head(8)

        if len(product_data) == 0:
            ax.text(0.5, 0.5, f"No {product} data", ha="center", va="center", transform=ax.transAxes)
            continue

        x = np.arange(len(product_data))
        width = 0.35

        ax.bar(x - width/2, product_data["actual_share"] * 100,
               width, label="Actual", color=colors["actual"], alpha=0.8)
        ax.bar(x + width/2, product_data["simulated_share"] * 100,
               width, label="Distance-Optimal", color=colors["simulated"], alpha=0.8)

        ax.set_xlabel("Port", fontsize=11)
        ax.set_ylabel("Share (%)", fontsize=11)
        ax.set_title(f"{product.upper()}", fontsize=13, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(product_data["port"], rotation=45, ha="right", fontsize=9)
        ax.legend()
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(DIR_PRELIM, "Port_Comparison_Chart.png"), dpi=300, bbox_inches="tight")
    print(f"\nChart saved: Port_Comparison_Chart.png")
    plt.show()

else:
    print("\nWarning: Chile_Port_Shares_Aduanas.csv not found")
    print("Can only show simulated shares, not comparison to actual")